In [1]:
prefix = '/Users/htelg'
overwrite = True
version_in = '1.0' # version of the cosine corrected files
serial_no = 649
frc_code = 'US_NOAA_MFR'
# pwv_corrfct = 1.42
p2fld_in = pl.Path(f'{prefix}/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_{str(serial_no)[-1]}/netcdf/v{version_in}/')
p2fld_out = pl.Path(f'{prefix}/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_{str(serial_no)[-1]}/aeronet_format/v{version_in}/')

In [2]:
wp = pd.DataFrame(p2fld_in.glob('*.nc'), columns=['p2f_in'])
wp.index = wp.apply(lambda row: pd.to_datetime(row.p2f_in.name.split('_')[-1].replace('.nc','')), axis =1)

wp['p2f_out'] = wp.apply(lambda row: p2fld_out / f"{frc_code}_{str(serial_no)[-1]}_{row.name:%Y%m%d}.aod", axis = 1)
wp.iloc[0].p2f_in, wp.iloc[0].p2f_out

(PosixPath('/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/netcdf/v1.0/aod_frc_649_v1.0_20251001.nc'),
 PosixPath('/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/aeronet_format/v1.0/US_NOAA_MFR_9_20251001.aod'))

In [3]:
# wp.iloc[0].p2f_out

In [4]:
wp

,p2f_in,p2f_out
2025-10-01,/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US...,/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US...
2025-10-11,/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US...,/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US...
2025-09-26,/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US...,/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US...
2025-10-21,/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US...,/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US...
2025-10-15,/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US...,/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US...
2025-10-04,/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US...,/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US...
2025-10-14,/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US...,/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US...
2025-10-10,/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US...,/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US...
2025-09-27,/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US...,/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US...
2025-09-28,/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US...,/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US...


In [5]:
# from pathlib import Path

# import numpy as np
# import pandas as pd
# import xarray as xr
def product(p2fin, p2fout):

    infile = p2fin#p2finp2fld_in / "aod_frc_648_v1.0_20251015.nc"
    frc_code = "FRC_CODE"
    angstrom_pair = "500_870"
    missing = -999
    
    ds = xr.open_dataset(infile)
    
    time = pd.DatetimeIndex(pd.to_datetime(ds.time.values))
    outfile = p2fout# infile.with_name(f"{frc_code}_{time[0]:%Y%m%d}.aod")
    p2fout.parent.mkdir(parents=True, exist_ok=True)
    
    def text(x):
        if isinstance(x, bytes):
            return x.decode()
        return str(x)
    
    
    def finite_or_nan(x):
        x = np.asarray(x, dtype=float)
        return np.where(np.isfinite(x), x, np.nan)
    
    
    def time_channel(name):
        if name not in ds or not {"time", "channel"}.issubset(ds[name].dims):
            return None
        return ds[name].transpose("time", "channel")
    
    
    def time_series(name):
        if name not in ds or "time" not in ds[name].dims:
            return None
        return ds[name].transpose("time")
    
    
    def pair_series(name):
        if name not in ds or "time" not in ds[name].dims:
            return None
    
        da = ds[name]
        pair_dim = next((dim for dim in da.dims if dim != "time"), None)
        if pair_dim is None:
            return da.transpose("time"), ""
    
        labels = [text(x) for x in ds[pair_dim].values]
        if angstrom_pair not in labels:
            return None
    
        return da.isel({pair_dim: labels.index(angstrom_pair)}).transpose("time")
    
    
    wavelengths = [int(x) for x in ds.channel.values]
    angstrom_label = angstrom_pair.replace("_", "-")
    
    table = pd.DataFrame(
        {
            "yyyy": time.strftime("%Y"),
            "mm": time.strftime("%m"),
            "dd": time.strftime("%d"),
            "HH(UTC)": time.strftime("%H"),
            "MM": time.strftime("%M"),
        }
    )
    
    aod = time_channel("aod")
    if aod is not None:
        for wl, values in zip(wavelengths, finite_or_nan(aod.values).T):
            table[f"AOD_{wl}nm"] = values
    
    angstrom_exponent = pair_series("angstrom_exponent")
    if angstrom_exponent is not None:
        table[f"{angstrom_label}_Angstrom_Exponent"] = finite_or_nan(angstrom_exponent.values)
    
    angstrom_turbidity = pair_series("angstrom_turbidity_coefficient")
    if angstrom_turbidity is not None:
        table[f"{angstrom_label}_Angstrom_Turbidity_Coefficient"] = finite_or_nan(angstrom_turbidity.values)
    
    aod_uncertainty = time_channel("aod_uncertainty")
    if aod_uncertainty is not None:
        for wl, values in zip(wavelengths, finite_or_nan(aod_uncertainty.values).T):
            table[f"AOD_Uncertainty_{wl}nm"] = values
    
    pressure = time_series("pressure")
    if pressure is not None:
        table["Atm.Pressure(hPa)"] = finite_or_nan(pressure.values)
    
    ozone = time_series("total_column_ozone")
    if ozone is not None:
        table["Ozone(Dobson)"] = finite_or_nan(ozone.values)
    
    no2 = time_series("total_column_no2")
    if no2 is not None:
        table["NO2(Dobson)"] = finite_or_nan(no2.values)
    
    keep = np.ones(time.size, dtype=bool)
    
    if aod is not None:
        keep &= np.isfinite(aod.values).any(axis=1)
    
    if "cloud_flag" in ds:
        keep &= ds.cloud_flag.values == 0
    
    table = table.loc[keep].reset_index(drop=True)
    
    with outfile.open("w", encoding="utf-8", newline="") as f:
        f.write("%wavelengths " + ",".join(map(str, wavelengths)) + "\n")
        table.to_csv(f, index=False, na_rep=str(missing), float_format="%.7g")
    
    return table

In [6]:
for idx, row in wp.iterrows():
    print(f'{row.p2f_in} -> {row.p2f_out} ...', end = '')
    out = product(row.p2f_in, row.p2f_out)
    print('done')
    # break

/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/netcdf/v1.0/aod_frc_649_v1.0_20251001.nc -> /Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/aeronet_format/v1.0/US_NOAA_MFR_9_20251001.aod ...

done
/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/netcdf/v1.0/aod_frc_649_v1.0_20251011.nc -> /Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/aeronet_format/v1.0/US_NOAA_MFR_9_20251011.aod ...done
/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/netcdf/v1.0/aod_frc_649_v1.0_20250926.nc -> /Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/aeronet_format/v1.0/US_NOAA_MFR_9_20250926.aod ...done
/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/netcdf/v1.0/aod_frc_649_v1.0_20251021.nc -> /Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/aeronet_format/v1.0/US_NOAA_MFR_9_20251021.aod ...done
/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/netcdf/v1.0/aod_frc_649_v1.0_20251015.nc -> /Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/aeronet_format/v1.0/US_NOAA_MFR_9_20251015.aod ...

done
/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/netcdf/v1.0/aod_frc_649_v1.0_20251004.nc -> /Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/aeronet_format/v1.0/US_NOAA_MFR_9_20251004.aod ...done
/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/netcdf/v1.0/aod_frc_649_v1.0_20251014.nc -> /Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/aeronet_format/v1.0/US_NOAA_MFR_9_20251014.aod ...done
/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/netcdf/v1.0/aod_frc_649_v1.0_20251010.nc -> /Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/aeronet_format/v1.0/US_NOAA_MFR_9_20251010.aod ...done
/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/netcdf/v1.0/aod_frc_649_v1.0_20250927.nc -> /Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/aeronet_format/v1.0/US_NOAA_MFR_9_20250927.aod ...

done
/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/netcdf/v1.0/aod_frc_649_v1.0_20250928.nc -> /Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/aeronet_format/v1.0/US_NOAA_MFR_9_20250928.aod ...done
/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/netcdf/v1.0/aod_frc_649_v1.0_20250929.nc -> /Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/aeronet_format/v1.0/US_NOAA_MFR_9_20250929.aod ...done
/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/netcdf/v1.0/aod_frc_649_v1.0_20251009.nc -> /Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/aeronet_format/v1.0/US_NOAA_MFR_9_20251009.aod ...done
/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/netcdf/v1.0/aod_frc_649_v1.0_20251019.nc -> /Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/aeronet_format/v1.0/US_NOAA_MFR_9_20251019.aod ...

done
/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/netcdf/v1.0/aod_frc_649_v1.0_20251008.nc -> /Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/aeronet_format/v1.0/US_NOAA_MFR_9_20251008.aod ...done
/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/netcdf/v1.0/aod_frc_649_v1.0_20251018.nc -> /Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/aeronet_format/v1.0/US_NOAA_MFR_9_20251018.aod ...done
/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/netcdf/v1.0/aod_frc_649_v1.0_20251007.nc -> /Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/aeronet_format/v1.0/US_NOAA_MFR_9_20251007.aod ...done
/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/netcdf/v1.0/aod_frc_649_v1.0_20251017.nc -> /Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/aeronet_format/v1.0/US_NOAA_MFR_9_20251017.aod ...

done
/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/netcdf/v1.0/aod_frc_649_v1.0_20251003.nc -> /Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/aeronet_format/v1.0/US_NOAA_MFR_9_20251003.aod ...done
/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/netcdf/v1.0/aod_frc_649_v1.0_20251013.nc -> /Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/aeronet_format/v1.0/US_NOAA_MFR_9_20251013.aod ...done
/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/netcdf/v1.0/aod_frc_649_v1.0_20251002.nc -> /Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/aeronet_format/v1.0/US_NOAA_MFR_9_20251002.aod ...done
/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/netcdf/v1.0/aod_frc_649_v1.0_20251012.nc -> /Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/aeronet_format/v1.0/US_NOAA_MFR_9_20251012.aod ...

done
/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/netcdf/v1.0/aod_frc_649_v1.0_20250925.nc -> /Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/aeronet_format/v1.0/US_NOAA_MFR_9_20250925.aod ...done
/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/netcdf/v1.0/aod_frc_649_v1.0_20251006.nc -> /Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/aeronet_format/v1.0/US_NOAA_MFR_9_20251006.aod ...done
/Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/netcdf/v1.0/aod_frc_649_v1.0_20251016.nc -> /Users/htelg/nfs/grad/campaign/frc/2025/AOD/US_NOAA_MFR_9/aeronet_format/v1.0/US_NOAA_MFR_9_20251016.aod ...done


NameError: name 'out' is not defined